In [6]:
import java.io.File

val file = File("sample_text")
val text = file.readText()

// was very useful
// https://docs.oracle.com/javase/8/docs/api/java/util/regex/Pattern.html

// also split by '\n' to avoid following
//      some sentence with number at the end 43.
//      41 other sentence with number at the start
// if '\n' split is not added, then we will have '43.\n41' instead of 2 separate entries
val rawSplitContent = text.split(" ", "\n").filter { it.isNotEmpty() }
// We don't want to bother with words like:
// 123123^^&1123123 (what?)
// 123.423.4321.54 (end of sentence, then floating point number, then again end of sentence?)
// 654,543,543  (might be just comma as sentence separation, or might be a floating point then comma?)

// in case potential splitted word ends with '.' or ','
// that means, it had whitspace characters around it.
// Therefore remove last trailing dot or comma to avoid
// skipping potential decimal. Example:
// "some text with kinda strange fractional at the end 43.51. Next sentence"
// here 43.51 should be normally processed
val splitContent = rawSplitContent.map { str ->
    str.trimEnd('.', ',')
}

// above text was split by whitespace characters.
// so use '^' and '$' (start and end of string) to check:
// 1. does potential number word have a sign?
// 2. does it start with digits?
// 3. does it have dot or comma?
// 4. if yes, do other digits follow it?
// 5. is it exponential notation? If so, check for 'e' or 'E'
val regexp = """^[+-]?\d+([.,]\d+)?([eE]\d+)?$""".toRegex()
// we may have text like: "some text with decimal at the end 43."
// then we should somehow understand that this last dot is full stop.
// Also, we might end up finding "some text with fractional at the end 43.21. Other text."

In [13]:
val numbersInText = mutableListOf<Double>()
val regexpMatches = splitContent.map { regexp.find(it) }.filterNotNull().map { it.value }

val (floatingPointRepresentationMatches, decimalRepresentationMatches) =  regexpMatches.partition {
    it.contains("""[.,eE]""".toRegex())
}

val decimals = decimalRepresentationMatches.map {
   try {
       it.toLong()
   } catch (e : NumberFormatException) {
       null
   }
}.filterNotNull()

val floatingPoints = floatingPointRepresentationMatches.map {
    try {
        it.toDouble()
    } catch (e : NumberFormatException) {
        null
    }
}.filterNotNull()

println("Decimals found:")
decimals.forEach {
    println(it)
}

println("Floating point numbers found:")
floatingPoints.forEach {
    println(it)
}


Decimals found:
123
12093
2093
32
14
2
5
3
2
321
Floating point numbers found:
43.32
-123.5312
4324.031
-4.121232300009E7
1.3E10
-2343.34
